In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from catboost import CatBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint, uniform

import os
import joblib

In [2]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [3]:
train_df = pd.read_csv("../data/train_preprocessed.csv")
test_df = pd.read_csv("../data/test_preprocessed.csv")

target_cols = ["CO2 Emission (Kg/m3)", "Compressive_strength"]

y_train = train_df[target_cols]
X_train = train_df.drop(columns=target_cols)

y_test = test_df[target_cols]
X_test = test_df.drop(columns=target_cols)

X_train.shape, X_test.shape

((513, 17), (129, 17))

# CatBoost

In [4]:
cbr = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    thread_count=1
)

model = MultiOutputRegressor(cbr)

param_dist = {
    "estimator__iterations": randint(200, 1200),
    "estimator__learning_rate": uniform(0.01, 0.29),
    "estimator__depth": randint(3, 11),
    "estimator__l2_leaf_reg": uniform(1.0, 9.0),
    "estimator__rsm": uniform(0.6, 0.4),
    "estimator__random_strength": uniform(0.0, 2.0),
    "estimator__border_count": randint(32, 255),
    "estimator__subsample": uniform(0.6, 0.4),
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best Hyperparameters: {'estimator__border_count': 78, 'estimator__depth': 5, 'estimator__iterations': 1022, 'estimator__l2_leaf_reg': 2.2451778317602122, 'estimator__learning_rate': 0.04849617245046124, 'estimator__random_strength': 1.939073734228318, 'estimator__rsm': 0.8858380416719809, 'estimator__subsample': 0.6164270067071503}
Best R2: 0.8703989131750796


In [5]:
best = random_search.best_params_

base_cbr = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False, 
    thread_count=1                
)

model = MultiOutputRegressor(base_cbr)

grid = {
    "estimator__iterations": sorted(set([
        max(50, best["estimator__iterations"] - 200),
        best["estimator__iterations"],
        best["estimator__iterations"] + 200,
    ])),

    "estimator__learning_rate": sorted(set([
        max(0.005, best["estimator__learning_rate"] * 0.85),
        best["estimator__learning_rate"],
        min(0.5, best["estimator__learning_rate"] * 1.15),
    ])),

    "estimator__depth": sorted(set([
        max(2, best["estimator__depth"] - 1),
        best["estimator__depth"],
        min(12, best["estimator__depth"] + 1),
    ])),

    "estimator__l2_leaf_reg": sorted(set([
        max(1e-6, best["estimator__l2_leaf_reg"] * 0.8),
        best["estimator__l2_leaf_reg"],
    ])),

    "estimator__rsm": sorted(set([
        max(0.1, best["estimator__rsm"] - 0.1),
        best["estimator__rsm"],
    ])),

    "estimator__random_strength": [best["estimator__random_strength"]],
    "estimator__border_count": [best["estimator__border_count"]],
    "estimator__subsample": [best.get("estimator__subsample", 0.8)],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

n_candidates = 1
for v in grid.values():
    n_candidates *= len(v)
print("Grid candidates:", n_candidates, "| Total fits:", n_candidates * cv.get_n_splits())

grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)


Grid candidates: 108 | Total fits: 1080
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best params (Grid): {'estimator__border_count': 78, 'estimator__depth': 5, 'estimator__iterations': 822, 'estimator__l2_leaf_reg': 1.7961422654081698, 'estimator__learning_rate': 0.04849617245046124, 'estimator__random_strength': 1.939073734228318, 'estimator__rsm': 0.8858380416719809, 'estimator__subsample': 0.6164270067071503}
Best R2 (Grid): 0.8728712190553629


In [6]:
best_model = grid_search.best_estimator_

cv = KFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_res = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=4,
    return_train_score=False
)

print("CV R2 mean ± std:", cv_res["test_r2"].mean(), "±", cv_res["test_r2"].std())
print("CV RMSE mean ± std:", (-cv_res["test_rmse"]).mean(), "±", (-cv_res["test_rmse"]).std())
print("CV MAE mean ± std:", (-cv_res["test_mae"]).mean(), "±", (-cv_res["test_mae"]).std())


CV R2 mean ± std: 0.8728712190553629 ± 0.04154575856753956
CV RMSE mean ± std: 6.901270883363812 ± 1.854662462712784
CV MAE mean ± std: 4.330282329785985 ± 0.855791458151194


In [7]:
y_pred = best_model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mae_test = mean_absolute_error(y_test, y_pred)

print("TEST RMSE:", rmse_test)
print("TEST MAE:", mae_test)

TEST RMSE: 6.930900249044391
TEST MAE: 4.244626485331558


In [8]:
r2_per_target = r2_score(y_test, y_pred, multioutput="raw_values")
rmse_per_target = np.sqrt(mean_squared_error(y_test, y_pred, multioutput="raw_values"))
mae_per_target = mean_absolute_error(y_test, y_pred, multioutput="raw_values")

for i, col in enumerate(y_test.columns):
    print(f"{col}: R2={r2_per_target[i]:.4f}, RMSE={rmse_per_target[i]:.4f}, MAE={mae_per_target[i]:.4f}")

CO2 Emission (Kg/m3): R2=0.9615, RMSE=6.8414, MAE=4.0187
Compressive_strength: R2=0.7979, RMSE=7.0192, MAE=4.4706


In [9]:
model_dir = "../ml_models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "catBoost_final_model.pkl")
joblib.dump(best_model, model_path)  

print(f"Model saved to: {model_path}")


Model saved to: ../ml_models\catBoost_final_model.pkl
